# CS 696 · Compare lookup and execution on matched interventions

Design a held-out comparison, score numerical answers separately from scope decisions and cited evidence, and identify a case that needs a reliability check.

Module 03 produced a traceable two-station simulator. To evaluate what the runner adds, give it and a historical lookup method the same frozen batches, process clauses and change cards. Decide how an unsupported request will be scored before looking at the predictions.

The museum’s amber batch has arrivals at 0, 1, 2 and 3 minutes. The jade batch has arrivals at 0, 15 and 30. Their recorded final labeling times under the original four-minute scans and seven-minute labels are 32 and 41. Each intervention is a separate hypothetical starting from the original batch.

## Hold out whole batches and their related questions

A paraphrase can carry the answer from one split into another. If two questions are about the same tray arrivals and the same underlying run, group them even when the wording or proposed intervention changes. Use the three-tray batch from Module 03 for development. Reserve both new batches, including every related change card and paraphrase, for this comparison. Both methods may read the historical record at evaluation time because the task explicitly provides it.

G(D) ∩ G(E) = ∅     G(S) = {g(q) : q ∈ S}

D is the development set, E the held-out evaluation set, and g(q) the original batch family behind question q. G(S) collects the families in split S. The empty intersection checks family separation. A family must follow the actual underlying batch even if someone renames a question.

In [1]:
questions = [
    ("prototype-change", "prototype", "development"),
    ("prototype-reworded", "prototype", "development"),
    ("A1", "amber", "evaluation"), ("A2", "amber", "evaluation"),
    ("J1", "jade", "evaluation"), ("J2", "jade", "evaluation"),
]
def shared_families(rows):
    dev = {family for _, family, split in rows if split == "development"}
    test = {family for _, family, split in rows if split == "evaluation"}
    return dev & test

assert not shared_families(questions)
leaked = questions + [("A1-another-phrasing", "amber", "development")]
assert shared_families(leaked) == {"amber"}
print("Families detected after the planted leak:", shared_families(leaked))

Families detected after the planted leak: {'amber'}


**Discuss** · If the same question appears with the tray colors renamed and the arrival order unchanged, what would determine its split?

Its source batch and generation history. A renamed copy of the amber instance stays with amber. The program detects overlap only when the family IDs are assigned correctly.

## Run two methods on the same four cards

The fixed lexical lookup baseline ranks historical finish records by the number of distinct words shared with the question, breaking ties by record ID. It returns the finish from the highest-ranked record. Both archive summaries contain “batch”, “final” and “finish”. The named batch picks the correct historical record. This deliberately simple baseline tests how often an archived number happens to work. The execution method applies the change to the original inputs and returns the last trace event. Both share a scope check. A request to add a worker needs an assignment rule absent from P1–P3, so both should abstain. The numeric reference for A1 was calculated separately: the scans finish at 4, 8, 12 and 16, and three-minute labels finish at 7, 11, 15 and 19.

ŷᴸ(q) = H[g(q)]     ŷˣ(q) = Fₙ(Iq(xg(q)))

H is the historical finish retrieved for the named batch in this fixture. Iq applies the intervention named in q to that batch’s inputs or settings. Fₙ reads the simulated last finish. L denotes lookup and X denotes execution. On A2, delaying the last arrival from 3 to 4 leaves its minute-12 scan start unchanged, so both methods return 32.

In [2]:
def simulate(arrivals, scan=4, label=7):
    if not arrivals or any(t < 0 for t in arrivals):
        raise ValueError("Provide nonnegative arrival minutes for at least one tray.")
    if list(arrivals) != sorted(arrivals) or min(scan, label) <= 0:
        raise ValueError("Order trays by arrival and use positive durations.")
    scan_free = label_free = 0
    trace = []
    for tray, arrival in enumerate(arrivals, start=1):
        s0 = max(arrival, scan_free)
        s1 = s0 + scan
        l0 = max(s1, label_free)
        l1 = l0 + label
        trace.append((tray, arrival, s0, s1, l0, l1))
        scan_free, label_free = s1, l1
    return trace

def run_change(arrivals, field, value):
    if field not in {"scan", "label", "delay_last"}:
        return None  # The supplied clauses do not define this change.
    if field == "delay_last":
        if value < 0:
            raise ValueError("Delay must be nonnegative.")
        changed = [*arrivals[:-1], arrivals[-1] + value]
        return simulate(changed)
    settings = {field: value}
    return simulate(arrivals, **settings)

batches = {
    "amber": {"arrivals": [0, 1, 2, 3], "historical_end": 32},
    "jade": {"arrivals": [0, 15, 30], "historical_end": 41},
}
# Each query has an ID, batch family, changed field and proposed value.
cards = [
    ("A1", "amber", "label", 3),
    ("A2", "amber", "delay_last", 1),
    ("J1", "jade", "scan", 2),
    ("J2", "jade", "extra_labeler", 1),
]
# Hand-checked targets are kept separate from inputs issued to the methods.
reference = {"A1": 19, "A2": 32, "J1": 39, "J2": None}

import re

def words(line):
    return set(re.findall(r"[a-z0-9]+", line.casefold()))

def lexical_finish(question):
    archive = {f"H-{f}": (f"{f} batch final label finish at minute "
                          f"{record['historical_end']}", record["historical_end"])
               for f, record in batches.items()}
    def order(eid):
        overlap = len(words(question) & words(archive[eid][0]))
        return (-overlap, eid)
    hit = min(archive, key=order)
    return archive[hit][1], {hit}, None

def predict(method, card):
    case, family, field, value = card
    batch = batches[family]
    if field not in {"scan", "label", "delay_last"}:
        return None, set(), None
    if method == "lookup":
        question = f"For {family} batch, final finish with {field} changed to {value}?"
        return lexical_finish(question)
    if method != "execute":
        raise ValueError("Unrecognized method")
    trace = run_change(batch["arrivals"], field, value)
    sources = {"P1", "P2", "P3", case}
    return trace[-1][-1], sources, trace

for card in cards:
    baseline = predict("lookup", card)
    tool = predict("execute", card)
    print(card[0], "lookup", baseline[0], "execute", tool[0], "target", reference[card[0]])
assert [predict("execute", q)[0] for q in cards] == [19, 32, 39, None]
assert [predict("lookup", q)[0] for q in cards] == [32, 32, 41, None]

A1 lookup 32 execute 19 target 19
A2 lookup 32 execute 32 target 32
J1 lookup 41 execute 39 target 39
J2 lookup None execute None target None


**Discuss** · Why keep the A2 question when both methods return the same number?

It is an unchanged-outcome control. Execution can show that the last tray was already waiting. Lookup reproduces the same number by repeating history. The numerical match alone cannot distinguish the two explanations.

## Separate the numeric result, the evidence and the scope decision

Grade each row against its independently checked answer card. For the three supported requests, check the exact final minute. Separately check that the source IDs include the process clauses and the intervention card. This is a citation-completeness test. A reader still has to inspect the cited clauses and trace to check the derivation. For the out-of-scope request, score whether the method withheld a number. Report coverage over all four requests so an abstention stays visible.

Aₙ = correct supported numbers / supported requests     C = numeric outputs / all requests

Aₙ counts a missing numerical prediction on a supported request as incorrect. C is numerical-answer coverage. Here there are three supported requests and one unsupported request. On supported requests, both methods return numbers for 3/3. On the complete set their numerical coverage is 3/4. Scope accuracy compares the decision to return a number with the reference scope on all four requests.

In [3]:
batches = {
    "amber": {"arrivals": [0, 1, 2, 3], "historical_end": 32},
    "jade": {"arrivals": [0, 15, 30], "historical_end": 41},
}
# Each query has an ID, batch family, changed field and proposed value.
cards = [
    ("A1", "amber", "label", 3),
    ("A2", "amber", "delay_last", 1),
    ("J1", "jade", "scan", 2),
    ("J2", "jade", "extra_labeler", 1),
]
# Hand-checked targets are kept separate from inputs issued to the methods.
reference = {"A1": 19, "A2": 32, "J1": 39, "J2": None}

# Frozen method outputs, shown as (answer in minutes, cited record IDs).
predictions = {
    "lookup": [(32, {"H-amber"}), (32, {"H-amber"}), (41, {"H-jade"}), (None, set())],
    "execute": [(y, {"P1", "P2", "P3", q}) if y is not None else (None, set())
                for q, y in [("A1", 19), ("A2", 32), ("J1", 39), ("J2", None)]],
}
for method, outputs in predictions.items():
    numeric = evidence = scope = returned = eligible = 0
    for card, (yhat, ids) in zip(cards, outputs):
        case, target = card[0], reference[card[0]]
        returned += yhat is not None
        scope += (target is None) == (yhat is None)
        if target is not None:
            eligible += 1
            numeric += yhat == target
            evidence += {"P1", "P2", "P3", case} <= ids
    print(method, f"numeric {numeric}/{eligible}, cited {evidence}/{eligible},",
          f"scope {scope}/{len(cards)}, coverage {returned}/{len(cards)}")

lookup numeric 1/3, cited 0/3, scope 4/4, coverage 3/4
execute numeric 3/3, cited 3/3, scope 4/4, coverage 3/4


**Discuss** · Suppose A1 reports 18 with all the expected IDs. Which checks would that answer pass?

It passes scope and citation completeness, then fails the numerical check. Inspecting the trace would show where the answer departed from the process. Conversely, the lookup baseline gets A2’s number right while citing only the original historical record.

## Independent task

Make a four-row audit ledger by running both methods. For each supported request record numerical correctness, citation completeness and the trace. For the unsupported request record whether the method abstained. Then inject two independent test faults into the execution outputs: remove the change-card citation from A1 and add one minute to J1’s final answer. Verify that your evaluator distinguishes them.

In [4]:
def simulate(arrivals, scan=4, label=7):
    if not arrivals or any(t < 0 for t in arrivals):
        raise ValueError("Provide nonnegative arrival minutes for at least one tray.")
    if list(arrivals) != sorted(arrivals) or min(scan, label) <= 0:
        raise ValueError("Order trays by arrival and use positive durations.")
    scan_free = label_free = 0
    trace = []
    for tray, arrival in enumerate(arrivals, start=1):
        s0 = max(arrival, scan_free)
        s1 = s0 + scan
        l0 = max(s1, label_free)
        l1 = l0 + label
        trace.append((tray, arrival, s0, s1, l0, l1))
        scan_free, label_free = s1, l1
    return trace

def run_change(arrivals, field, value):
    if field not in {"scan", "label", "delay_last"}:
        return None  # The supplied clauses do not define this change.
    if field == "delay_last":
        if value < 0:
            raise ValueError("Delay must be nonnegative.")
        changed = [*arrivals[:-1], arrivals[-1] + value]
        return simulate(changed)
    settings = {field: value}
    return simulate(arrivals, **settings)

batches = {
    "amber": {"arrivals": [0, 1, 2, 3], "historical_end": 32},
    "jade": {"arrivals": [0, 15, 30], "historical_end": 41},
}
# Each query has an ID, batch family, changed field and proposed value.
cards = [
    ("A1", "amber", "label", 3),
    ("A2", "amber", "delay_last", 1),
    ("J1", "jade", "scan", 2),
    ("J2", "jade", "extra_labeler", 1),
]
# Hand-checked targets are kept separate from inputs issued to the methods.
reference = {"A1": 19, "A2": 32, "J1": 39, "J2": None}

import re

def words(line):
    return set(re.findall(r"[a-z0-9]+", line.casefold()))

def lexical_finish(question):
    archive = {f"H-{f}": (f"{f} batch final label finish at minute "
                          f"{record['historical_end']}", record["historical_end"])
               for f, record in batches.items()}
    def order(eid):
        overlap = len(words(question) & words(archive[eid][0]))
        return (-overlap, eid)
    hit = min(archive, key=order)
    return archive[hit][1], {hit}, None

def predict(method, card):
    case, family, field, value = card
    batch = batches[family]
    if field not in {"scan", "label", "delay_last"}:
        return None, set(), None
    if method == "lookup":
        question = f"For {family} batch, final finish with {field} changed to {value}?"
        return lexical_finish(question)
    if method != "execute":
        raise ValueError("Unrecognized method")
    trace = run_change(batch["arrivals"], field, value)
    sources = {"P1", "P2", "P3", case}
    return trace[-1][-1], sources, trace

outputs = {card[0]: predict("execute", card) for card in cards}
for card in cards:
    lookup_value, lookup_ids, _ = predict("lookup", card)
    tool_value, tool_ids, _ = outputs[card[0]]
    print(card[0], "lookup", lookup_value, sorted(lookup_ids),
          "execute", tool_value, sorted(tool_ids))
# Make a copy of outputs for each fault. Keep the original trace for comparison.

A1 lookup 32 ['H-amber'] execute 19 ['A1', 'P1', 'P2', 'P3']
A2 lookup 32 ['H-amber'] execute 32 ['A2', 'P1', 'P2', 'P3']
J1 lookup 41 ['H-jade'] execute 39 ['J1', 'P1', 'P2', 'P3']
J2 lookup None [] execute None []


## Compare your solution

Record the reason for a failure beside its Boolean check. A citation fault can leave the arithmetic correct. A numerical fault can leave the citations and the stored station events intact.

In [5]:
def simulate(arrivals, scan=4, label=7):
    if not arrivals or any(t < 0 for t in arrivals):
        raise ValueError("Provide nonnegative arrival minutes for at least one tray.")
    if list(arrivals) != sorted(arrivals) or min(scan, label) <= 0:
        raise ValueError("Order trays by arrival and use positive durations.")
    scan_free = label_free = 0
    trace = []
    for tray, arrival in enumerate(arrivals, start=1):
        s0 = max(arrival, scan_free)
        s1 = s0 + scan
        l0 = max(s1, label_free)
        l1 = l0 + label
        trace.append((tray, arrival, s0, s1, l0, l1))
        scan_free, label_free = s1, l1
    return trace

def run_change(arrivals, field, value):
    if field not in {"scan", "label", "delay_last"}:
        return None  # The supplied clauses do not define this change.
    if field == "delay_last":
        if value < 0:
            raise ValueError("Delay must be nonnegative.")
        changed = [*arrivals[:-1], arrivals[-1] + value]
        return simulate(changed)
    settings = {field: value}
    return simulate(arrivals, **settings)

batches = {
    "amber": {"arrivals": [0, 1, 2, 3], "historical_end": 32},
    "jade": {"arrivals": [0, 15, 30], "historical_end": 41},
}
# Each query has an ID, batch family, changed field and proposed value.
cards = [
    ("A1", "amber", "label", 3),
    ("A2", "amber", "delay_last", 1),
    ("J1", "jade", "scan", 2),
    ("J2", "jade", "extra_labeler", 1),
]
# Hand-checked targets are kept separate from inputs issued to the methods.
reference = {"A1": 19, "A2": 32, "J1": 39, "J2": None}

import re

def words(line):
    return set(re.findall(r"[a-z0-9]+", line.casefold()))

def lexical_finish(question):
    archive = {f"H-{f}": (f"{f} batch final label finish at minute "
                          f"{record['historical_end']}", record["historical_end"])
               for f, record in batches.items()}
    def order(eid):
        overlap = len(words(question) & words(archive[eid][0]))
        return (-overlap, eid)
    hit = min(archive, key=order)
    return archive[hit][1], {hit}, None

def predict(method, card):
    case, family, field, value = card
    batch = batches[family]
    if field not in {"scan", "label", "delay_last"}:
        return None, set(), None
    if method == "lookup":
        question = f"For {family} batch, final finish with {field} changed to {value}?"
        return lexical_finish(question)
    if method != "execute":
        raise ValueError("Unrecognized method")
    trace = run_change(batch["arrivals"], field, value)
    sources = {"P1", "P2", "P3", case}
    return trace[-1][-1], sources, trace

def check(card, output):
    case, target = card[0], reference[card[0]]
    yhat, citations, trace = output
    if target is None:
        return {"scope": yhat is None}
    return {"scope": yhat is not None, "number": yhat == target,
            "citations": {"P1", "P2", "P3", case} <= citations,
            "matches_trace": bool(trace) and yhat == trace[-1][-1]}

for card in cards:
    for method in ["lookup", "execute"]:
        result = predict(method, card)
        print(card[0], method, check(card, result), "trace", result[2])

a1, j1 = cards[0], cards[2]
y, ids, trace = predict("execute", a1)
missing = check(a1, (y, ids - {"A1"}, trace))
assert missing == {"scope": True, "number": True,
                   "citations": False, "matches_trace": True}
y, ids, trace = predict("execute", j1)
shifted = check(j1, (y + 1, ids, trace))
assert shifted == {"scope": True, "number": False,
                   "citations": True, "matches_trace": False}
print("A1 citation fault:", missing)
print("J1 numerical fault:", shifted)

A1 lookup {'scope': True, 'number': False, 'citations': False, 'matches_trace': False} trace None
A1 execute {'scope': True, 'number': True, 'citations': True, 'matches_trace': True} trace [(1, 0, 0, 4, 4, 7), (2, 1, 4, 8, 8, 11), (3, 2, 8, 12, 12, 15), (4, 3, 12, 16, 16, 19)]
A2 lookup {'scope': True, 'number': True, 'citations': False, 'matches_trace': False} trace None
A2 execute {'scope': True, 'number': True, 'citations': True, 'matches_trace': True} trace [(1, 0, 0, 4, 4, 11), (2, 1, 4, 8, 11, 18), (3, 2, 8, 12, 18, 25), (4, 4, 12, 16, 25, 32)]
J1 lookup {'scope': True, 'number': False, 'citations': False, 'matches_trace': False} trace None
J1 execute {'scope': True, 'number': True, 'citations': True, 'matches_trace': True} trace [(1, 0, 0, 2, 2, 9), (2, 15, 15, 17, 17, 24), (3, 30, 30, 32, 32, 39)]
J2 lookup {'scope': True} trace None
J2 execute {'scope': True} trace None
A1 citation fault: {'scope': True, 'number': True, 'citations': False, 'matches_trace': True}
J1 numerical f

On this four-card exercise, lookup gets 1/3 supported numbers correct and execution gets 3/3. Both withhold a number for J2. The two injected faults show that arithmetic and citation completeness test different properties. A trace can still be consistent with an incorrectly translated program, a possibility carried into the next module.

## Mentor review

Bring one frozen batch-level comparison ledger with case IDs, original arrivals, change cards, historical finishes, both predictions, source IDs and traces. Ask the mentor to accept the grouped comparison as evidence about these two batches or identify a specific evaluation flaw.

- Related questions share a split and both methods receive the same case information.
- The report gives numerical correctness on three eligible cards, scope decisions on four and coverage on four.
- The unchanged outcome, unsupported change and two fault injections each have a traceable interpretation.
- Which case looks successful if you score only the final number?
- What additional independent check would distinguish a correct rule from a simulator tuned to reproduce history?

Pass Module 05 the original clauses, four frozen intervention cards, historical finish per batch, hand-checked reference answers, method outputs and station traces. Preserve the original and fault-injected versions separately. The next review can ask whether agreement with a historical run is enough to trust an intervention.